# DocuDevs: Classify Products into a Category Tree

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/docudevs/python-examples/blob/main/10-product-classification.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-View_Source-blue?logo=github)](https://github.com/docudevs/python-examples)

File new grocery products into a category tree. A records case holds labelled products as evidence, the classifier files held-out products and flags the uncertain ones, a reviewer's correction becomes a memo instruction, and a holdout evaluation measures whether that instruction helps.

**What you'll learn:**
- Create a records dataset and import labelled records
- Upload and activate a category tree
- Classify products and read accuracy per tree level
- Work with flagged rows and review results
- Turn a correction into an approved memo instruction
- Measure the instruction with a holdout evaluation

**Requirements:** a DocuDevs API key and `docu-devs-api-client>=1.11.0`, the first release with case records.

## The Use Case

A retailer keeps a category tree for its range: *Dairy > Cheese > Hard cheese*, *Beverages > Soft drinks > Carbonated drinks*, and so on. New products arrive from suppliers without a category, and each one has to be filed into the tree. The retailer already has thousands of products with a category, and those labels show how the team files things.

DocuDevs turns the labelled products into a **records** knowledge base. For a new product, the classifier looks at the most similar labelled products as evidence and chooses a leaf of the tree. When it is unsure, it flags the row for review. A reviewer's correction can become a **memo** instruction, which a person approves before it is used. A **holdout evaluation** then checks whether the instruction improves accuracy on products the case has not seen.

The labelled products are from [Open Food Facts](https://world.openfoodfacts.org), a free food product database. The [data README](data/product-classification/README.md) explains how the files were prepared and their licence.

## Setup

Install the SDK. Colab already includes many packages, so the shared dependencies are installed first and the SDK is installed without changing them.

In [ ]:
%pip install -q --upgrade "pydantic>=2.0,<=2.12.3" "httpx>=0.28.1,<1.0" "attrs>=21.3.0" "python-dateutil>=2.8.0" "click>=8.0.0"
%pip install -q --upgrade --no-deps "docu-devs-api-client>=1.11.0"

Next, connect to the API. Store your key in Colab Secrets as `DOCUDEVS_API_KEY` (recommended), in an environment variable of the same name, or set `API_KEY` below. The knobs control the classify, evaluation and agent steps. The defaults use the MINI model tier and a small token budget.

In [ ]:
import csv
import io
import json
import os
import random
import urllib.request
from importlib.metadata import version as package_version
from pathlib import Path

REQUIRED_SDK = (1, 11, 0)
installed_sdk = package_version("docu-devs-api-client")
if tuple(int(part) for part in installed_sdk.split(".")[:3]) < REQUIRED_SDK:
    raise RuntimeError(
        f"docu-devs-api-client {installed_sdk} is older than 1.11.0, which adds case records. "
        "Run the install cell again, then restart the runtime."
    )
print(f"docu-devs-api-client {installed_sdk}")

from docudevs import DocuDevsClient

try:
    from google.colab import userdata
    API_KEY = userdata.get("DOCUDEVS_API_KEY")
except Exception:
    API_KEY = os.getenv("DOCUDEVS_API_KEY", "your-api-key-here")

if not API_KEY or API_KEY == "your-api-key-here":
    raise ValueError("Set your DOCUDEVS_API_KEY - get one at https://docudevs.ai")

client = DocuDevsClient(token=API_KEY)
print("Connected to DocuDevs API")

LLM_LEVEL = "MINI"
TARGET_DEPTH = [1, 2, 3]             # tree levels reported with each classify result
CLASSIFY_BUDGET_TOKENS = 600_000
HOLDOUT_EVAL_SAMPLE = 100
RUN_AGENT = False                    # True starts the optional headless investigation
DELETE_CASE_AT_END = True

## 1. Load the Data

The notebook reads three small files from `data/product-classification/` in this repository, or from GitHub when it runs in Colab:

- `tree.json` is the category tree. Each node has a dotted `code` such as `1.2.6`, the `parent` code (`null` for a root) and an English `name`.
- `records.csv` holds 2,500 labelled products. The `leaf` column is the label: the code of the product's most specific category. A label is always a leaf of the tree.
- `holdout.csv` holds 100 products with the same columns. The classifier never sees their `leaf`; it is the answer we compare against.

The EAN (`ean`) is kept as text, so codes with leading zeros are not changed.

In [ ]:
DATA_PATH = Path("data/product-classification")
RAW_URL = "https://raw.githubusercontent.com/docudevs/python-examples/main/data/product-classification"


def read_data_file(name):
    local_file = DATA_PATH / name
    if local_file.exists():
        return local_file.read_text(encoding="utf-8")
    with urllib.request.urlopen(f"{RAW_URL}/{name}", timeout=120) as response:
        return response.read().decode("utf-8")


tree = json.loads(read_data_file("tree.json"))
records = list(csv.DictReader(io.StringIO(read_data_file("records.csv"))))
holdout = list(csv.DictReader(io.StringIO(read_data_file("holdout.csv"))))

nodes_by_code = {node["code"]: node for node in tree}
parent_codes = {node["parent"] for node in tree}
leaves = [node["code"] for node in tree if node["code"] not in parent_codes]
max_depth = max(len(code.split(".")) for code in leaves)
records_by_ean = {row["ean"]: row for row in records}
truth_by_key = {row["ean"]: row["leaf"] for row in holdout}
print(f"{len(tree)} categories, {len(leaves)} leaves, {len(records):,} records, {len(holdout)} holdout products")

Each record is a product with a label. Here are two records and the path of the category their label names:

In [ ]:
def category_path(code):
    path = []
    while code:
        path.append(nodes_by_code[code]["name"])
        code = nodes_by_code[code]["parent"]
    return " > ".join(reversed(path))


for row in records[:2]:
    print(f"{row['name']} ({row['brand'] or 'no brand'}, {row['quantity'] or 'no size'})")
    print(f"  label {row['leaf']}: {category_path(row['leaf'])}")
print("Root categories:", ", ".join(node["name"] for node in tree if node["parent"] is None))

## 2. Create a Case and Import the Records

A records case holds only records: there are no documents and no extraction step. The dataset setting tells the case how to read each record:

- `key` is the record's unique id, here the EAN.
- `text` maps columns to the text fields the classifier reads (`TEXT_FIELDS`).
- `label` is the column that holds the answer, here the leaf code.
- `metadata` keeps values with the record without searching them, here the brand.
- `embedded_text` is the text that is embedded for similarity search. Empty fields are skipped.

In [ ]:
from docudevs.models.cases_controller_create_case_request import CasesControllerCreateCaseRequest

TEXT_FIELDS = ("name", "generic_name", "brand", "quantity", "ingredients")
FIELD_MAPPING = {
    "key": "ean",
    "text": {field: field for field in TEXT_FIELDS},
    "label": "leaf",
    "metadata": {"brand": "brand"},
}
EMBEDDED_TEXT = {
    "template": "{name}\n{generic_name}\n{brand}\n{quantity}\n{ingredients}",
    "skipEmptyFields": True,
}


def get_case_id(response):
    if response.parsed is not None:
        return response.parsed.id
    return json.loads(response.content.decode())["id"]


created = await client.create_case(
    body=CasesControllerCreateCaseRequest(name="Product classification (notebook)")
)
if created.status_code not in (200, 201):
    raise RuntimeError(f"Error creating case: {created.status_code} {created.content.decode()}")
case_id = get_case_id(created)
print(f"Created case {case_id}")

await client.set_dataset(
    case_id,
    FIELD_MAPPING,
    embedded_text=EMBEDDED_TEXT,
    index_settings={"topK": 10, "hybrid": False},
    classify_defaults={"targetDepth": TARGET_DEPTH},
)
print("Dataset configured")

Now import the 2,500 records. Imports run in the background, so the cell waits until the import is complete and then prints its counts.

In [ ]:
imported = await client.bulk_upsert_records(case_id, records)
final = await client.wait_for_import(case_id, imported["importId"], timeout=1800, poll_interval=5.0)
if final["status"] != "COMPLETED":
    raise RuntimeError(f"Import ended with {final['status']}")
print({key: final.get(key) for key in ("rowsReceived", "rowsRejected", "recordsAdded")})

## 3. Upload and Activate the Tree

Trees are versioned. Uploading creates a draft, and activating makes it the case's active tree. Activation checks that every record label exists in the tree, so a tree that is missing a label is rejected.

In [ ]:
uploaded = await client.upload_tree(case_id, tree, name="Open Food Facts categories")
version_id = uploaded["treeVersionId"]
await client.activate_tree(case_id, version_id, wait=True, timeout=1800)
print(f"Tree version {version_id} is active with {len(leaves)} leaves")

## 4. Classify the Held-Out Products

The held-out products are sent inline, without their labels. The job pins the tree, memo and records versions when it starts, so later changes do not affect a running job. Each result reports a category at every level in `TARGET_DEPTH`; level 2 of `1.2.6` is `1.2`.

Accuracy is measured per level by comparing code prefixes. A row the model declined counts as wrong.

In [ ]:
def inline_row(row):
    text = {field: row[field] for field in TEXT_FIELDS if row[field]}
    result = {"key": row["ean"], "text": text}
    if row["brand"]:
        result["metadata"] = {"brand": row["brand"]}
    return result


classify_config = {
    "llmLevel": LLM_LEVEL,
    "concurrency": 4,
    "maxTotalTokens": CLASSIFY_BUDGET_TOKENS,
    "targetDepth": TARGET_DEPTH,
}
accepted = await client.classify(case_id, rows=[inline_row(row) for row in holdout], config=classify_config)
job_id = int(accepted["jobId"])
job = await client.wait_for_classify(case_id, job_id, timeout=3600, poll_interval=5.0)
print(f"Classify job {job_id}: {job['status']}", job.get("counts"))

results = [row async for row in client.get_classify_results(case_id, job_id)]
declined = sum(1 for row in results if not row.get("chosen"))
print(f"{len(results)} rows, {declined} declined or failed")

In [ ]:
def prefix(code, depth):
    return ".".join(code.split(".")[:depth])


def accuracy_by_level(results, truth_by_key, levels):
    scores = {}
    for depth in levels:
        right = 0
        for row in results:
            chosen = (row.get("chosen") or {}).get("code")
            if chosen and prefix(chosen, depth) == prefix(truth_by_key[row["key"]], depth):
                right += 1
        scores[depth] = (right, len(results))
    return scores


levels = sorted(set(TARGET_DEPTH) | {max_depth})
for depth, (right, total) in accuracy_by_level(results, truth_by_key, levels).items():
    print(f"Level {depth}: {right}/{total} correct ({right / total:.0%})")

### Flagged rows

The classifier flags a row for review when it is unsure, for example when two categories fit about equally well. Each flagged row names its reasons and the records it used as evidence. Here are the first few:

In [ ]:
flagged = [row for row in results if (row.get("review") or {}).get("required")]
print(f"{len(flagged)} of {len(results)} rows need review")

for row in flagged[:5]:
    chosen = row.get("chosen") or {}
    print(f"\nRow {row['rowId']}: {row['key']} (historical label {truth_by_key[row['key']]})")
    print(f"  Chosen: {' > '.join(chosen.get('path', [])) or 'declined'}")
    print(f"  Reasons: {', '.join(row['review']['reasons'])}")
    for ean in row.get("evidenceKeys", [])[:3]:
        evidence = records_by_ean.get(ean, {})
        print(f"  Evidence: {evidence.get('name', ean)} -> {evidence.get('leaf', '?')}")

## 5. Review a Sample of Records

Review works on records that are already in the case, so we classify a small batch of record keys and compare the results with the labels the records already have. The batch is drawn from the records, not from the holdout, so the review does not touch the holdout.

- `accept` confirms a correct result and writes it as the record's review label.
- `correct` writes the category you supply and adds a correction memo entry with your reason.

In [ ]:
REVIEW_SEED = 20261013
REVIEW_BATCH = 30
review_rows = random.Random(REVIEW_SEED).sample(records, REVIEW_BATCH)
review_accepted = await client.classify(
    case_id, keys=[row["ean"] for row in review_rows], config=classify_config
)
review_job_id = int(review_accepted["jobId"])
await client.wait_for_classify(case_id, review_job_id, timeout=3600, poll_interval=5.0)
review_results = [row async for row in client.get_classify_results(case_id, review_job_id)]


def agrees_with_label(row):
    return row["chosen"]["code"] == records_by_ean[row["key"]]["leaf"]


agree = [row for row in review_results if row.get("chosen") and agrees_with_label(row)]
disagree = [row for row in review_results if row.get("chosen") and not agrees_with_label(row)]
print(f"Review batch: {len(agree)} agree with the record label, {len(disagree)} disagree")

Writes to labels are asynchronous, so the helper below waits for the label write before we read the record back. The first agreeing row is accepted:

In [ ]:
async def wait_for_label_write(review):
    write = review.get("labelWrite") or {}
    if write.get("importId") is not None:
        final = await client.wait_for_import(case_id, write["importId"], timeout=1800, poll_interval=2.0)
        if final["status"] != "COMPLETED":
            raise RuntimeError(f"Label write ended {final['status']}")


accept_row = agree[0]
accepted_review = await client.review_classify(case_id, review_job_id, accept_row["rowId"], "accept")
await wait_for_label_write(accepted_review)
accepted_label = (await client.get_record(case_id, accept_row["key"])).get("label")
print(f"Accepted {accept_row['key']}: label is {accepted_label}")

The first disagreeing row is corrected to the record's label. The reason is stored with the correction memo entry.

In [ ]:
correct_row = disagree[0]
historical_code = records_by_ean[correct_row["key"]]["leaf"]
corrected_review = await client.review_classify(
    case_id,
    review_job_id,
    correct_row["rowId"],
    "correct",
    code=historical_code,
    reason="Reviewed: the product name and ingredients describe the historical category, not the one the model chose.",
)
await wait_for_label_write(corrected_review)
corrected_label = (await client.get_record(case_id, correct_row["key"])).get("label")
print(f"Corrected {correct_row['key']}: model said {correct_row['chosen']['code']}, review label is {corrected_label}")

## 6. Memo: Write an Instruction from the Correction

The correction shows the model choosing one category when the reviewer expects a sibling. We turn that into a memo **instruction** that covers the subtree of the parent category. The instruction starts as `proposed` and changes nothing until a person approves it.

In [ ]:
truth_code = historical_code
predicted_code = correct_row["chosen"]["code"]
truth_name = nodes_by_code[truth_code]["name"]
predicted_name = nodes_by_code[predicted_code]["name"]
rule_text = (
    f"Between '{truth_name}' and '{predicted_name}': file a product under '{truth_name}' "
    f"when its name or generic name describes that product type. Use '{predicted_name}' only "
    "when the product name names that type."
)
parent = nodes_by_code[truth_code]["parent"] or truth_code
entry = await client.add_memo_entry(case_id, "instruction", {"kind": "subtree", "code": parent}, rule_text)
approved = await client.approve_memo_entry(case_id, entry["id"])
print(f"Approved memo entry {entry['id']} (status {approved.get('status')})")
print(rule_text)

### Holdout evaluation with and without the memo

The [holdout evaluation](https://docs.docudevs.ai/advanced/case-evals) is a server-side test. It draws its own sample of labelled records (not the holdout list above) and keeps near-duplicates out of each sampled record's evidence. Both variants use that sample: `without-memo` ignores the memo, and `with-memo` uses the active memo, which now includes the instruction.

The rule came from a review correction, not from the evaluation sample. One caveat: the correction is a memo entry for one record, so if that record is in the sample, its own result reflects the correction.

In [ ]:
variants = [
    {
        "name": "without-memo",
        "config": {"llmLevel": LLM_LEVEL, "targetDepth": TARGET_DEPTH, "memo": {"use": "none"}},
    },
    {"name": "with-memo"},
]
started = await client.start_holdout_eval(
    case_id,
    variants,
    sample_size=HOLDOUT_EVAL_SAMPLE,
    seed=7,
    config={"llmLevel": LLM_LEVEL, "targetDepth": TARGET_DEPTH},
)
finished = await client.wait_for_eval(case_id, started["evalId"], timeout=3600, poll_interval=5.0)

report = finished.get("report") or {}
print("Grouping:", (report.get("sample") or {}).get("grouping"))
for variant in report.get("variants") or []:
    target = ((variant.get("score") or {}).get("targetAccuracy")) or {}
    print(f"{variant.get('name')}: target accuracy {target.get('all')} over {target.get('nAll')} rows")
print("Paired differences:")
print(json.dumps(report.get("comparisons"), indent=2)[:2000])

## 7. Optional: Investigate the Flagged Rows with the Agent

A headless run works through the flagged rows of the classify job with a small token budget. It cannot confirm anything: its proposals wait in the pending-action inbox for a person. Set `RUN_AGENT = True` to start it. The [chat agent](https://docs.docudevs.ai/advanced/case-agent) can also answer questions about a single row.

In [ ]:
if RUN_AGENT:
    started_run = await client.start_agent_run(
        case_id,
        "investigate-flagged",
        params={"job_id": job_id, "max_rows": 3},
        budget={"max_total_tokens": 20_000},
        origin_job_id=job_id,
    )
    run = await client.wait_for_agent_run(case_id, started_run["runId"], timeout=1800, poll_interval=5.0)
    print(run["status"], "| tokens:", run.get("tokensUsed"), "| note:", run.get("error"))
    print((run.get("report") or {}).get("summary"))
    for action in (await client.list_agent_actions(case_id)).get("items", []):
        print(action.get("status"), "-", action.get("summary"))
else:
    print("Set RUN_AGENT = True to start a headless investigation (20k token budget).")

## 8. Cleanup

Deleting the case removes its records, results, memo and evaluations. Set `DELETE_CASE_AT_END = False` to keep the case and explore it in the web app.

In [ ]:
if DELETE_CASE_AT_END:
    await client.delete_case(case_id)
    print(f"Deleted case {case_id}")
else:
    print(f"Kept case {case_id}")

## Next Steps

- [Records datasets](https://docs.docudevs.ai/advanced/records-datasets): field mapping, imports and search
- [Classification](https://docs.docudevs.ai/advanced/classification): candidate strategy, flags and routing
- [Case memo](https://docs.docudevs.ai/advanced/case-memo) and [holdout evaluation](https://docs.docudevs.ai/advanced/case-evals)
- [Headless agent runs](https://docs.docudevs.ai/advanced/agent-runs)

Data: [Open Food Facts](https://world.openfoodfacts.org), ODbL 1.0 and DbCL 1.0. Contains data from Open Food Facts, available under the Open Database License. See the [data README](data/product-classification/README.md).